# WSA_99 — Phase Overview

**Purpose.** Consolidate actual Weather Sensitivity outputs into the final phase overview.

> Run from the project repository. Outputs are generated only from the project data and frozen model artifacts.

In [ ]:
# Import libraries
from pathlib import Path
import sys, json, pandas as pd, numpy as np, matplotlib.pyplot as plt


In [2]:
# Define config paths
PROJECT_ROOT = Path.cwd()
while not (PROJECT_ROOT / "src").exists() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
CONFIG = PROJECT_ROOT / "configs" / "weather_sensitivity.yaml"
CONFIG

WindowsPath('e:/jcuenca/OneDrive - GUSCanada/5toTerm/01_Capstone/DataLocal/ontario-electricity-peak-risk/configs/weather_sensitivity.yaml')

In [3]:
# Import modules for weather_sensitivity
from src.ontario_peak_risk.weather_sensitivity.common import load_config, ensure_dirs

In [4]:
cfg, project_root = load_config(CONFIG)
paths = ensure_dirs(cfg, project_root)
print("Project root:", project_root)

Project root: E:\jcuenca\OneDrive - GUSCanada\5toTerm\01_Capstone\DataLocal\ontario-electricity-peak-risk


In [5]:
r = pd.read_parquet(paths["outputs_dir"] / "WSA_sensitivity_master.parquet")
summary = pd.read_csv(paths["outputs_dir"] / "WSA_04_fsa_scenario_summary.csv")
scenario = (
    r.groupby(["scenario", "temperature_delta_c"], observed=True)
    .agg(
        mean_abs_forecast_delta_kwh=("forecast_delta_kwh", lambda s: s.abs().mean()),
        max_abs_forecast_delta_kwh=("forecast_delta_kwh", lambda s: s.abs().max()),
        mean_abs_peak_risk_delta=("peak_risk_delta", lambda s: s.abs().mean()),
        max_abs_peak_risk_delta=("peak_risk_delta", lambda s: s.abs().max()),
        alert_changes=("alert_changed", "sum"),
        out_of_range=(
            "temperature_domain_status",
            lambda s: (s == "OUT_OF_RANGE").sum(),
        ),
    )
    .reset_index()
)
display(scenario)

,scenario,temperature_delta_c,mean_abs_forecast_delta_kwh,max_abs_forecast_delta_kwh,mean_abs_peak_risk_delta,max_abs_peak_risk_delta,alert_changes,out_of_range
0,+2.5C,2.5,48.947027,205.984713,0.052406,0.425148,5,0
1,+5C,5.0,80.608473,293.257204,0.105733,0.596323,17,0
2,-2.5C,-2.5,89.276406,536.642827,0.037970,0.273067,6,0
3,-5C,-5.0,226.359455,774.130697,0.056483,0.407749,8,0
4,Baseline,0.0,0.000000,0.000000,0.000000,0.000000,0,0


In [7]:
lines = [
    "# Weather Sensitivity Analysis — Phase Overview",
    "",
    "## Scope",
    "Frozen Model v1 sensitivity to controlled perturbations of `origin__Temp (°C)`. Only temperature is changed; all other model features remain fixed. Models are not retrained. Results are not causal and are not target-hour future-weather sensitivity.",
    "",
    "## Scenarios",
    "−5°C, −2.5°C, Baseline, +2.5°C, +5°C.",
    "",
    "## Scenario-level results",
    "",
    scenario.to_markdown(index=False),
    "",
    "## Validation",
    f"- Total scenario predictions: {len(r):,}",
    f"- OUT_OF_RANGE rows: {(r.temperature_domain_status == 'OUT_OF_RANGE').sum():,}",
    f"- CAUTION rows: {(r.temperature_domain_status == 'CAUTION').sum():,}",
    f"- Alert decision changes: {int(r.alert_changed.sum()):,}",
    "",
    "## Interpretation notes",
    "- Forecast deltas are measured against the exact Baseline feature frame.",
    "- Peak-Risk deltas use the same frozen XGBoost score and official 0.06 alert threshold.",
    "- Scenario asymmetry is evidence of nonlinear model response, not causality.",
    "- OOD scenario predictions should be interpreted with additional caution.",
    "",
    "## Main artifacts",
    "- `WSA_sensitivity_master.parquet/csv`",
    "- `WSA_02_rf_forecast_sensitivity.csv`",
    "- `WSA_03_xgb_peak_risk_sensitivity.csv`",
    "- `WSA_04_fsa_scenario_summary.csv`",
    "- `WSA_04_horizon_summary.csv`",
    "- `WSA_04_symmetry_analysis.csv`",
    "- comparative figures under `reports/figures/weather_sensitivity/`",
]
out = "".join(lines)
(paths["docs_dir"] / "Weather_Sensitivity_Analysis_Overview.md").write_text(
    out, encoding="utf-8"
)
print(out)
print("WSA_99 RESULT: COMPLETE")

# Weather Sensitivity Analysis — Phase Overview## ScopeFrozen Model v1 sensitivity to controlled perturbations of `origin__Temp (°C)`. Only temperature is changed; all other model features remain fixed. Models are not retrained. Results are not causal and are not target-hour future-weather sensitivity.## Scenarios−5°C, −2.5°C, Baseline, +2.5°C, +5°C.## Scenario-level results| scenario   |   temperature_delta_c |   mean_abs_forecast_delta_kwh |   max_abs_forecast_delta_kwh |   mean_abs_peak_risk_delta |   max_abs_peak_risk_delta |   alert_changes |   out_of_range |
|:-----------|----------------------:|------------------------------:|-----------------------------:|---------------------------:|--------------------------:|----------------:|---------------:|
| +2.5C      |                   2.5 |                       48.947  |                      205.985 |                  0.0524064 |                  0.425148 |               5 |              0 |
| +5C        |                   5   |   

In [9]:

# ---------------------------------------------------------
# Load final Weather Sensitivity Analysis artifacts
# ---------------------------------------------------------

master_path = paths["outputs_dir"] / "WSA_sensitivity_master.parquet"
fsa_summary_path = paths["outputs_dir"] / "WSA_04_fsa_scenario_summary.csv"
horizon_summary_path = paths["outputs_dir"] / "WSA_04_horizon_summary.csv"
symmetry_path = paths["outputs_dir"] / "WSA_04_symmetry_analysis.csv"

required_files = [
    master_path,
    fsa_summary_path,
    horizon_summary_path,
    symmetry_path,
]

missing_files = [str(p) for p in required_files if not p.exists()]

if missing_files:
    raise FileNotFoundError(
        "Required Weather Sensitivity artifacts are missing:\n"
        + "\n".join(missing_files)
    )

results = pd.read_parquet(master_path)
fsa_summary = pd.read_csv(fsa_summary_path)
horizon_summary = pd.read_csv(horizon_summary_path)
symmetry = pd.read_csv(symmetry_path)

# ---------------------------------------------------------
# Structural validation
# ---------------------------------------------------------

expected_rows = (
    len(cfg["analysis"]["fsas"])
    * int(cfg["analysis"]["horizons"])
    * len(cfg["analysis"]["scenarios_c"])
)

if len(results) != expected_rows:
    raise ValueError(
        f"Unexpected number of scenario predictions: "
        f"{len(results)}; expected {expected_rows}."
    )

# ---------------------------------------------------------
# Scenario-level summary
# ---------------------------------------------------------

scenario_summary = (
    results
    .groupby(
        ["scenario", "temperature_delta_c"],
        observed=True
    )
    .agg(
        mean_abs_forecast_delta_kwh=(
            "forecast_delta_kwh",
            lambda s: s.abs().mean()
        ),
        max_abs_forecast_delta_kwh=(
            "forecast_delta_kwh",
            lambda s: s.abs().max()
        ),
        mean_abs_peak_risk_delta=(
            "peak_risk_delta",
            lambda s: s.abs().mean()
        ),
        max_abs_peak_risk_delta=(
            "peak_risk_delta",
            lambda s: s.abs().max()
        ),
        alert_changes=(
            "alert_changed",
            "sum"
        ),
        out_of_range=(
            "temperature_domain_status",
            lambda s: (s == "OUT_OF_RANGE").sum()
        ),
    )
    .reset_index()
    .sort_values("temperature_delta_c")
)

# ---------------------------------------------------------
# Validation statistics
# ---------------------------------------------------------

out_of_range_rows = (
    results["temperature_domain_status"]
    .eq("OUT_OF_RANGE")
    .sum()
)

caution_rows = (
    results["temperature_domain_status"]
    .eq("CAUTION")
    .sum()
)

alert_changes = (
    results["alert_changed"]
    .astype(bool)
    .sum()
)

# Largest absolute RF response
largest_idx = results["forecast_delta_kwh"].abs().idxmax()
largest = results.loc[largest_idx]

# ---------------------------------------------------------
# Build Markdown table without relying on pandas.to_markdown()
# ---------------------------------------------------------

def markdown_table(df, columns, labels=None, decimals=3):
    labels = labels or columns

    header = "| " + " | ".join(labels) + " |"
    separator = "| " + " | ".join(["---"] * len(columns)) + " |"

    rows = []

    for _, row in df.iterrows():
        values = []

        for col in columns:
            value = row[col]

            if isinstance(value, float):
                value = f"{value:.{decimals}f}"

            values.append(str(value))

        rows.append("| " + " | ".join(values) + " |")

    return "\n".join([header, separator] + rows)


scenario_table = markdown_table(
    scenario_summary,
    columns=[
        "scenario",
        "temperature_delta_c",
        "mean_abs_forecast_delta_kwh",
        "max_abs_forecast_delta_kwh",
        "mean_abs_peak_risk_delta",
        "max_abs_peak_risk_delta",
        "alert_changes",
        "out_of_range",
    ],
    labels=[
        "Scenario",
        "Δ Temperature (°C)",
        "Mean ABS(Δ Forecast) (kWh)",
        "Max ABS(Δ Forecast) (kWh)",
        "Mean ABS(Δ Peak-Risk)",
        "Max ABS(Δ Peak-Risk)",
        "Alert Changes",
        "Out of Range",
    ],
)

# ---------------------------------------------------------
# Key findings
# ---------------------------------------------------------

cold_5 = scenario_summary.loc[
    scenario_summary["temperature_delta_c"].eq(-5.0)
].iloc[0]

warm_5 = scenario_summary.loc[
    scenario_summary["temperature_delta_c"].eq(5.0)
].iloc[0]

cold_25 = scenario_summary.loc[
    scenario_summary["temperature_delta_c"].eq(-2.5)
].iloc[0]

warm_25 = scenario_summary.loc[
    scenario_summary["temperature_delta_c"].eq(2.5)
].iloc[0]

# ---------------------------------------------------------
# Generate final Overview
# ---------------------------------------------------------

overview = f"""# Weather Sensitivity Analysis — Phase Overview

## 1. Scope

This phase evaluates the sensitivity of the frozen Model v1 to controlled
perturbations of `origin__Temp (°C)`.

Only the temperature feature is modified. All other model inputs remain fixed,
and neither the Random Forest demand-forecasting models nor the XGBoost
Peak-Risk models are retrained.

The analysis therefore measures **model sensitivity**, not causal weather
effects. It also does not represent target-hour future-weather sensitivity,
because Model v1 uses weather observed at the forecast origin.

## 2. Scenarios

Five controlled temperature scenarios were evaluated:

- −5.0 °C
- −2.5 °C
- Baseline
- +2.5 °C
- +5.0 °C

Each scenario was evaluated across:

- 6 FSAs
- 24 forecast horizons
- Random Forest electricity-demand forecasts
- XGBoost Peak-Risk scores and alert decisions

Total scenario predictions: **{len(results):,}**

## 3. Scenario-Level Results

{scenario_table}

## 4. Key Findings

### Demand Forecast Sensitivity

The Random Forest forecasts respond meaningfully to changes in forecast-origin
temperature.

The average absolute forecast response was:

- **{cold_25["mean_abs_forecast_delta_kwh"]:.2f} kWh** for −2.5 °C
- **{cold_5["mean_abs_forecast_delta_kwh"]:.2f} kWh** for −5.0 °C
- **{warm_25["mean_abs_forecast_delta_kwh"]:.2f} kWh** for +2.5 °C
- **{warm_5["mean_abs_forecast_delta_kwh"]:.2f} kWh** for +5.0 °C

The response is not perfectly symmetric between warmer and colder scenarios,
indicating nonlinear behavior in the fitted forecasting model.

### Peak-Risk Sensitivity

Temperature perturbations also modify the XGBoost Peak-Risk score.

Across all scenarios, **{alert_changes} alert decisions** changed relative to
the Baseline scenario using the frozen operational threshold of
**{cfg["analysis"]["official_peak_threshold"]:.2f}**.

This demonstrates that temperature variation can influence not only the
continuous risk score but, in some FSA/horizon combinations, the final
operational alert decision.

### FSA and Horizon Differences

Sensitivity is heterogeneous across both geography and forecast horizon.
Some FSA/horizon combinations respond substantially more strongly than others.

The largest absolute demand-forecast response observed was
**{abs(largest["forecast_delta_kwh"]):.2f} kWh**, for:

- FSA: **{largest["fsa"]}**
- Horizon: **h+{int(largest["horizon"])}**
- Scenario: **{largest["scenario"]}**

This supports evaluating sensitivity at the FSA/horizon level rather than
relying only on province-wide averages.

### Nonlinearity and Asymmetry

The ±2.5 °C and ±5 °C scenarios do not generate perfectly mirrored model
responses.

This asymmetry is evidence of nonlinear model behavior and interactions among
the frozen model features. It must not be interpreted as evidence of a causal
relationship between temperature and electricity demand or Peak-Risk.

## 5. Domain Validation

Temperature-domain validation produced:

- Total scenario predictions: **{len(results):,}**
- NORMAL: **{len(results) - caution_rows - out_of_range_rows:,}**
- CAUTION: **{caution_rows:,}**
- OUT_OF_RANGE: **{out_of_range_rows:,}**

All evaluated scenarios therefore remained within the validated temperature
support of Model v1 for this operational forecast origin.

## 6. Interpretation Boundaries

The following limitations apply:

1. The analysis perturbs only `origin__Temp (°C)`.
2. All other model features remain fixed.
3. Model parameters are frozen.
4. Results represent model response, not causal effects.
5. The analysis does not simulate complete alternative weather trajectories.
6. Target-hour future weather is not currently part of Model v1.
7. The official Peak-Risk decision threshold remains fixed at
   **{cfg["analysis"]["official_peak_threshold"]:.2f}**.

A future Model v2 could incorporate target-hour weather forecasts and permit
full future-weather scenario analysis.

## 7. Decision-Support Relevance

The sensitivity analysis provides an additional decision-support layer beyond
the baseline forecast.

The final application can therefore present:

- baseline 24-hour demand forecasts;
- baseline Peak-Risk probabilities and alerts;
- controlled temperature scenarios;
- changes in expected electricity demand;
- changes in Peak-Risk score;
- alert transitions relative to baseline;
- warnings when scenarios approach or exceed the model-development domain.

This functionality should be interpreted as **scenario-based model
sensitivity**, rather than deterministic prediction under alternative weather
conditions.

## 8. Main Artifacts

- `WSA_sensitivity_master.parquet`
- `WSA_sensitivity_master.csv`
- `WSA_02_rf_forecast_sensitivity.csv`
- `WSA_03_xgb_peak_risk_sensitivity.csv`
- `WSA_03_alert_transitions.csv`
- `WSA_04_fsa_scenario_summary.csv`
- `WSA_04_horizon_summary.csv`
- `WSA_04_symmetry_analysis.csv`
- `WSA_06_validation_summary.csv`
- `WSA_06_nonlinearity_symmetry.csv`
- comparative figures under `reports/figures/weather_sensitivity/`

## 9. Phase Status

**Weather Sensitivity Analysis: COMPLETE**

The outputs are ready for integration into the final Decision-Support
Application.
"""

overview_path = (
    paths["docs_dir"]
    / "Weather_Sensitivity_Analysis_Overview.md"
)

overview_path.write_text(
    overview,
    encoding="utf-8"
)

print("Overview saved to:")
print(overview_path)
print()
print("WSA_99 RESULT: COMPLETE")

Overview saved to:
E:\jcuenca\OneDrive - GUSCanada\5toTerm\01_Capstone\DataLocal\ontario-electricity-peak-risk\docs\weather_sensitivity\Weather_Sensitivity_Analysis_Overview.md

WSA_99 RESULT: COMPLETE
